# 🎯 Hyperparameter Tuning

This notebook contains the logic for Bayesian Hyperparameter Optimization used during the development of the 2026 NCAA prediction models. We use `scikit-optimize` (`skopt`) to search for the best parameters for XGBoost, Random Forest, and Logistic Regression.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import make_scorer, brier_score_loss
from skopt import BayesSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import warnings

warnings.filterwarnings('ignore')

OUTPUT_PATH = '../output'
rs_data = pd.read_csv(f'{OUTPUT_PATH}/RegularDataModel.csv')
tourney_data = pd.read_csv(f'{OUTPUT_PATH}/TournamentDataModel.csv')

## 1. Why Bayesian Optimization?

Unlike Grid Search, which exhaustively tests all combinations, or Random Search, which picks points at random, **Bayesian Optimization** builds a probability model of the objective function and uses it to select the most promising hyperparameters to evaluate in each step. This significantly reduces the time required to find optimal settings.

We minimize the **Negative Brier Score** (since `skopt` maximizes the scorer).

## 2. Search Spaces and Rationale

### XGBoost
- **`n_estimators`**: (50, 300). Constrained to prevent over-learning on noisy tournament data.
- **`learning_rate`**: (0.01, 0.3, 'log-uniform'). Log-uniform helps explore small learning rates more effectively.
- **`max_depth`**: (3, 8). Shallow trees are preferred to avoid fitting to specific year-by-year anomalies.

### Random Forest
- **`min_samples_split` & `min_samples_leaf`**: (1, 20). Higher values ensure that each split generalizes to a larger chunk of the data.

### Logistic Regression
- **`C`**: (1e-3, 1e2, 'log-uniform'). Broad range for regularization strength.

In [ ]:
def tune_models(season_data, features):
    """
    Uses BayesSearchCV to optimize model hyperparameters.
    """
    brier_scorer = make_scorer(brier_score_loss, greater_is_better=False, response_method='predict_proba')

    X_train = season_data[features]
    y_train = season_data['Pred']
    
    models_to_tune = {
        'xgb': {
            'model': XGBClassifier(random_state=42),
            'params': {
                'n_estimators': (50, 300),
                'learning_rate': (0.01, 0.3, 'log-uniform'),
                'max_depth': (3, 8),
                'subsample': (0.5, 1.0),
                'colsample_bytree': (0.5, 1.0)
            }
        },
        'rf': {
            'model': RandomForestClassifier(random_state=42),
            'params': {
                'n_estimators': (50, 300),
                'max_depth': (3, 12),
                'min_samples_split': (2, 20),
                'min_samples_leaf': (1, 20)
            }
        },
        'lr': {
            'model': LogisticRegression(solver='liblinear', random_state=42, max_iter=1000),
            'params': {
                'C': (1e-3, 1e2, 'log-uniform'),
                'penalty': ['l1', 'l2']
            }
        }
    }
    
    best_estimators = {}
    
    for name, config in models_to_tune.items():
        print(f"\nTuning {name}...")
        bayes_search = BayesSearchCV(
            estimator=config['model'],
            search_spaces=config['params'],
            n_iter=20, 
            cv=3, 
            scoring=brier_scorer,
            n_jobs=-1,
            random_state=42
        )
        
        bayes_search.fit(X_train, y_train)
        best_estimators[name] = bayes_search.best_estimator_
        
    return best_estimators